# Import a CSV into `ig_market_data.db`

Loads one CSV written by `01_download_prices.ipynb`
(`data/ig_<epic-slug>_<resolution-suffix>_<timestamp>.csv`) into the
per-resolution `candles_<suffix>` table of `data/ig_market_data.db`, created
here via `candle_db.init_candles_table()` (the schema single source of
truth) and filled with `INSERT OR IGNORE`
keyed on `UNIQUE(epic, snapshot_time_utc)`, so re-running is idempotent.

The CSV holds `snapshot_time_utc` + the flattened `{open,high,low,close}_{bid,ask,mid}_price`
and `last_traded_volume` columns — everything the table stores except `epic`,
which isn't in the CSV and is taken from `config.py` (`.env`), i.e. whatever
`01_download_prices.ipynb` was configured for when it wrote the CSV. Override
`EPIC` / `RESOLUTION` in the config cell if you're importing a CSV from a
different market or resolution (`RESOLUTION` only picks the target table — it
isn't a stored column).

**Prerequisite:** the `igmarket` package installed — `pip install -e ".[dev]"`
from the repo root (see the README). Otherwise standard-library only.

## 1. Config

`DB_PATH`, `EPIC`, `RESOLUTION` come from `config.py`'s `Config` (same as
`03_view_ig_prices.ipynb`). `CSV_PATH` defaults to the newest `data/ig_*.csv`;
set it explicitly to import a specific file.

In [1]:
import csv
import sqlite3
from pathlib import Path

from igmarket.candle_db import table_name_for_resolution
from igmarket.config import Config

cfg = Config.from_env()

# epic + resolution are not in the CSV - take them from config.py / .env (what
# 01_download_prices.ipynb used to write it). Override if importing another market.
EPIC = cfg.epic
RESOLUTION = cfg.resolution
DB_PATH = cfg.db_path

# CSV to import: newest data/ig_*.csv by mtime. Set explicitly to pick another.
_candidates = sorted(cfg.data_dir.glob("ig_*.csv"), key=lambda p: p.stat().st_mtime)
assert _candidates, "No data/ig_*.csv found - run 01_download_prices.ipynb first, or set CSV_PATH."
CSV_PATH = _candidates[-1]

print(f"CSV_PATH   = {CSV_PATH}")
print(f"DB_PATH    = {DB_PATH}")
print(f"EPIC       = {EPIC!r}")
print(f"RESOLUTION = {RESOLUTION!r}  -> table {table_name_for_resolution(RESOLUTION)}")

CSV_PATH   = C:\repos\github.com\jupyter-notebooks\data\ig_nasdaq_daily_20260906213458.csv
DB_PATH    = C:\repos\github.com\jupyter-notebooks\data\ig_market_data.db
EPIC       = 'IX.D.NASDAQ.IFA.IP'
RESOLUTION = 'DAY'  -> table candles_1d


## 2. Read + check the CSV

Fail loudly unless the columns are exactly `constants/csv_headers.py`'s
`CSV_HEADERS` — this notebook only imports CSVs shaped by
`01_download_prices.ipynb`.

In [2]:
from igmarket.constants.csv_headers import CSV_HEADERS

with open(CSV_PATH, newline="") as fh:
    reader = csv.DictReader(fh)
    header = reader.fieldnames
    csv_rows = list(reader)

assert header == CSV_HEADERS, (
    f"Unexpected CSV columns.\n  got:      {header}\n  expected: {CSV_HEADERS}\n"
    "This notebook only imports CSVs written by 01_download_prices.ipynb."
)
print(f"{CSV_PATH.name}: {len(csv_rows)} data row(s), columns OK")
for row in csv_rows[:3]:
    print(dict(row))

ig_nasdaq_daily_20260906213458.csv: 1768 data row(s), columns OK
{'snapshot_time_utc': '2020-12-30T13:00:00', 'open_bid_price': '12893.2', 'open_ask_price': '12895.2', 'open_mid_price': '12894.2', 'high_bid_price': '12912.9', 'high_ask_price': '12913.9', 'high_mid_price': '12913.4', 'low_bid_price': '12826.2', 'low_ask_price': '12827.2', 'low_mid_price': '12826.7', 'close_bid_price': '12860.1', 'close_ask_price': '12862.1', 'close_mid_price': '12861.1', 'last_traded_volume': '312275'}
{'snapshot_time_utc': '2020-12-31T13:00:00', 'open_bid_price': '12859.6', 'open_ask_price': '12861.6', 'open_mid_price': '12860.6', 'high_bid_price': '12901.5', 'high_ask_price': '12902.5', 'high_mid_price': '12902.0', 'low_bid_price': '12803.7', 'low_ask_price': '12804.7', 'low_mid_price': '12804.2', 'close_bid_price': '12887.0', 'close_ask_price': '12892.0', 'close_mid_price': '12889.5', 'last_traded_volume': '247713'}
{'snapshot_time_utc': '2021-01-03T13:00:00', 'open_bid_price': '12907.2', 'open_ask_p

## 3. Import into `candles_<suffix>`

Each CSV row is rebuilt into a raw-IG-style candle dict, then flattened with
the producer's own `candle_csv.candle_to_row()` and inserted via
`candle_db.INSERT_COLUMNS` — the same flattening
`01_download_prices.ipynb` uses for its CSV, so imported rows match it exactly.

In [3]:
from igmarket.candle_csv import candle_to_row
from igmarket.candle_db import INSERT_COLUMNS, init_candles_table
from igmarket.constants.ig_candle_fields import CandleField, PriceField


def _float(x):
    x = (x or "").strip()
    return float(x) if x else None


def _int(x):
    x = (x or "").strip()
    return int(x) if x else None


def csv_row_to_candle(r):
    """Rebuild a raw-IG-style candle dict from one CSV row - the shape
    `candle_csv.candle_to_row()` expects to flatten."""

    def node(prefix):
        return {
            PriceField.BID: _float(r[f"{prefix}_bid_price"]),
            PriceField.ASK: _float(r[f"{prefix}_ask_price"]),
        }

    return {
        CandleField.SNAPSHOT_TIME_UTC: r["snapshot_time_utc"],
        CandleField.OPEN_PRICE: node("open"),
        CandleField.HIGH_PRICE: node("high"),
        CandleField.LOW_PRICE: node("low"),
        CandleField.CLOSE_PRICE: node("close"),
        CandleField.LAST_TRADED_VOLUME: _int(r["last_traded_volume"]),
    }


conn = sqlite3.connect(DB_PATH)
table = init_candles_table(conn, RESOLUTION)  # CREATE TABLE IF NOT EXISTS - candle_db's schema

insert_rows = [(EPIC, *candle_to_row(csv_row_to_candle(r))) for r in csv_rows]

before = conn.total_changes
placeholders = ", ".join("?" for _ in INSERT_COLUMNS)
conn.executemany(
    f"INSERT OR IGNORE INTO {table} ({', '.join(INSERT_COLUMNS)}) VALUES ({placeholders})",
    insert_rows,
)
conn.commit()
inserted = conn.total_changes - before
print(f"{DB_PATH}: +{inserted} new row(s) in `{table}` "
      f"({len(insert_rows) - inserted} already present, skipped by INSERT OR IGNORE)")

C:\repos\github.com\jupyter-notebooks\data\ig_market_data.db: +0 new row(s) in `candles_1d` (1768 already present, skipped by INSERT OR IGNORE)


## 4. Verify

Read the rows back for this `epic` with `candle_db.load_candles` — the same
function `03_view_ig_prices.ipynb` uses.

In [4]:
from igmarket.candle_db import load_candles

rows = list(load_candles(conn, RESOLUTION, epic=EPIC))
conn.close()

print(f"{len(rows)} candle(s) in `{table}` for {EPIC}")
if rows:
    print("first:", rows[0])
    print("last: ", rows[-1])

1768 candle(s) in `candles_1d` for IX.D.NASDAQ.IFA.IP
first: {'epic': 'IX.D.NASDAQ.IFA.IP', 'snapshot_time_utc': '2020-12-30T13:00:00', 'open': 12894.2, 'high': 12913.4, 'low': 12826.7, 'close': 12861.1, 'open_mid_price': 12894.2, 'high_mid_price': 12913.4, 'low_mid_price': 12826.7, 'close_mid_price': 12861.1, 'volume': 312275}
last:  {'epic': 'IX.D.NASDAQ.IFA.IP', 'snapshot_time_utc': '2026-09-04T14:00:00', 'open': 29647.0, 'high': 29648.0, 'low': 29439.0, 'close': 29490.5, 'open_mid_price': 29647.0, 'high_mid_price': 29648.0, 'low_mid_price': 29439.0, 'close_mid_price': 29490.5, 'volume': 294097}
